# JIT Compilation + Tracing + XLA

This notebook explores JAX's Just-In-Time compilation system, tracing mechanism, and the XLA compiler that transforms Python code into optimized machine code.

## What is JIT Compilation?

### 🧠 The Core Concept

**Python execution:**
- Line by line interpretation
- Dynamic but slow

**JAX JIT approach:**
```
Python → trace → build computation graph → compile → run optimized version
```

This enables massive speedups by bypassing the Python interpreter and running compiled code directly on hardware.

In [1]:
# Import JAX libraries
import jax
import jax.numpy as jnp

### ✅ Exercise 31 — JIT Compile a Function

Let's start with a simple function and see how JAX compiles it.

In [2]:
def cube(x):
    return x ** 3

In [3]:
# JAX uses lazy compilation - it doesn't compile immediately
cube_jit = jax.jit(cube)

### ✅ Exercise 32 — First Run (With Compilation Overhead)

The first call triggers compilation, so it takes longer.

In [4]:
# First execution - triggers tracing and compilation
cube_jit(10.0)  # Lazy compilation happens here

Array(1000., dtype=float32, weak_type=True)

## What Happens on First Run?

1. **JAX traces** the function
2. **Builds computation graph** (JAXPR)
3. **Compiles with XLA** to machine code
4. **Executes compiled code**

This compilation overhead is why the first run is slower.

### ✅ Exercise 33 — Second Run (Fast)

Subsequent calls use the compiled code and are much faster.

## Why is the Second Run Faster?

- **No tracing** - already done
- **No compiling** - cached executable
- **Just runs compiled machine code** directly on device

In [5]:
cube_jit(10.24)  # Uses cached compiled version

Array(1073.7418, dtype=float32, weak_type=True)

## 🧠 The Power of JIT

**Why is it faster?**
- **Python interpreter bypassed** - no overhead
- **Compiled code runs directly** on device (CPU/GPU/TPU)
- **Hardware optimizations** - vectorization, fusion, parallelization

This is the core performance advantage of JAX.

### ✅ Exercise 34 — Store Result

Store the result in a variable for later use.

In [6]:
cube_value = cube_jit(10.24)
cube_value

Array(1073.7418, dtype=float32, weak_type=True)

### ✅ Exercise 35 — Disable JIT

Sometimes you want to run the pure Python version for debugging.

In [7]:
# Run without JIT compilation - pure Python execution
with jax.disable_jit():
    cube_value_nojit = cube_jit(10.24)

cube_value_nojit

1073.7418240000002

### ✅ Exercise 36 — Get Output Shape Without Execution

Use `eval_shape` to determine output shape/dtype without running the computation.

In [8]:
# Get output shape without executing the computation
cube_shape = jax.eval_shape(cube_jit, 10.24)
cube_shape  # Scalar output

ShapeDtypeStruct(shape=(), dtype=float32, weak_type=True)

In [9]:
# Works with complex shapes too
jax.eval_shape(lambda x: x**2, jnp.ones((1000, 1000)))

ShapeDtypeStruct(shape=(1000, 1000), dtype=float32)

## The Complete JAX Compilation Flow
Let's explore the full compilation pipeline from Python to machine code.

In [10]:
# Import JAX libraries
import jax
import jax.numpy as jnp

# Our simple Python function - this is the HIGH LEVEL starting point
def f(x, y):
    return jnp.sin(x) + y**2  # JAX will trace this to build its internal representation

# Sample concrete inputs (same shapes JAX will trace with)
x = jnp.ones((2, 2))  # 2x2 matrix of ones
y = 2 * jnp.ones((2, 2))  # 2x2 matrix of twos

print("=== 1. JAXPR (High-level IR from tracing) ===")
# STEP 1: Get JAXPR - this shows what JAX "understands" about your function
# make_jaxpr traces your Python function and produces JAX's internal computation graph
jaxpr = jax.make_jaxpr(f)(x, y)
print(jaxpr)
# This looks like functional-style pseudocode of your computation

print("\n=== 2. Lowered IR → StableHLO (Low-level IR) ===")
# STEP 2: Lower to StableHLO - this is the MLIR dialect XLA compilers understand
# jit(f) creates a compiled version, .lower() shows intermediate representations
lowered = jax.jit(f).lower(x, y)

print("Full MLIR with StableHLO operations (human readable):")
print(lowered.as_text())  # Shows the full pipeline: JAXPR → MLIR → stablehlo ops

print("\nJust the StableHLO dialect (what goes to XLA):")
stablehlo_module = lowered.compiler_ir(dialect="stablehlo")
print(stablehlo_module)  # Pure StableHLO - portable across backends!

print("\n=== 3. XLA Compilation → Machine Code Executable ===")
# STEP 3: Actually compile to machine code (CPU/GPU kernels)
executable = lowered.compile()  # This runs XLA compiler on StableHLO
print(f"Executable created! Running it gives: {executable(x, y)}")

print("\n=== SUMMARY ===")
print("Python → (trace) → JAXPR → (lower) → StableHLO → (XLA compile) → Machine Code ✓")

=== 1. JAXPR (High-level IR from tracing) ===
{ lambda ; a:f32[2,2] b:f32[2,2]. let
    c:f32[2,2] = sin a
    d:f32[2,2] = integer_pow[y=2] b
    e:f32[2,2] = add c d
  in (e,) }

=== 2. Lowered IR → StableHLO (Low-level IR) ===
Full MLIR with StableHLO operations (human readable):
module @jit_f attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<2x2xf32>, %arg1: tensor<2x2xf32>) -> (tensor<2x2xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.sine %arg0 : tensor<2x2xf32>
    %1 = stablehlo.multiply %arg1, %arg1 : tensor<2x2xf32>
    %2 = stablehlo.add %0, %1 : tensor<2x2xf32>
    return %2 : tensor<2x2xf32>
  }
}


Just the StableHLO dialect (what goes to XLA):
module @jit_f attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<2x2xf32>, %arg1: tensor<2x2xf32>) -> (tensor<2x2xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.sine %arg0 : tensor<2x2xf32>


### ✅ Exercise 37 — Examine JAXPR
JAXPR is JAX's intermediate representation - a functional description of your computation.

In [11]:
cube_jaxpr = jax.make_jaxpr(cube_jit)(10.24)
cube_jaxpr

{ lambda ; a:f32[]. let
    b:f32[] = jit[
      name=cube
      jaxpr={ lambda ; a:f32[]. let b:f32[] = integer_pow[y=3] a in (b,) }
    ] a
  in (b,) }

### ✅ Exercise 38 — Examine XLA HLO

HLO (High-Level Optimizer) is the intermediate representation that XLA compiler uses.

In [12]:
# Lower and get HLO - this is what XLA actually compiles
xla_ir = cube_jit.lower(10.24).compiler_ir()
print(xla_ir)

module @jit_cube attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<f32>) -> (tensor<f32> {jax.result_info = "result"}) {
    %0 = stablehlo.multiply %arg0, %arg0 : tensor<f32>
    %1 = stablehlo.multiply %0, %arg0 : tensor<f32>
    return %1 : tensor<f32>
  }
}



## Understanding the Intermediate Representations

JAX traces your function and converts it into pure operations. Each operation (multiply, add, etc.) is typed and explicit. This IR (Intermediate Representation) is what XLA compiler sends to GPU/TPU/CPU for execution.

JIT compilation allows the accelerator to:
- **Fuse operations** - combine multiple steps
- **Optimize memory** - reduce temporary allocations
- **Parallelize** - execute in parallel when possible

### ✅ Exercise 39 — Benchmark JIT vs Non-JIT

Let's measure the actual performance difference.

In [13]:
import time

def benchmark(fn, x, runs=100):
    start = time.perf_counter()
    for _ in range(runs):
        jax.block_until_ready(fn(x))
    return (time.perf_counter() - start) / runs

x = jnp.array(10.24)

# Warm up JIT cache
cube_jit(x)

jit_time    = benchmark(cube_jit, x)
nojit_time  = benchmark(cube, x)

print(f"JIT    : {jit_time * 1e6:.2f} µs")
print(f"No JIT : {nojit_time * 1e6:.2f} µs")
print(f"Speedup: {nojit_time / jit_time:.2f}x")

# jax.block_until_ready forces async dispatch to complete.
# Without it, JAX returns a future — timing would be misleading.

JIT    : 5.54 µs
No JIT : 180.66 µs
Speedup: 32.60x


### ✅ Exercise 40 — Handle Dynamic Shapes with static_argnames

When function behavior depends on Python values (not arrays), mark them as static.

In [14]:
import jax.numpy as jnp
from jax import jit

@jit
def power(x, exp):
    return jnp.ones(exp) * x  # shape depends on exp

try:
    power(10.24, exp=3)  # works because 3 is Python int
    power(10.24, exp=jnp.array(3))  # ❌ fails without static_argnames
except Exception as TypeError:
    print("Error occurred, mark exp as static")
    import functools
    import jax
    import jax.numpy as jnp
    
    @functools.partial(jax.jit, static_argnames=("exp",))
    def power_static(x, exp):
        return jnp.ones(exp) * x

    print(power_static(10.24, exp=3))   # ✅ works

Error occurred, mark exp as static
[10.24 10.24 10.24]
